In [6]:
import mysql.connector
import psycopg2
import docx
import pandas as pd
from datetime import date
from docx.enum.text import WD_ALIGN_PARAGRAPH
import json

inputFacilityID = input("Enter facility ID")

# ----------- 1. Connect to MySQL and PostgreSQL--------------

# a. Load configs from config.json
with open("config.json") as f:
    config=json.load(f)
    
# b. Connect to MySQL Server
mydb = mysql.connector.connect(
  **config["mysql"]
)
mySQLcursor = mydb.cursor()


# c. Connect to PostgreSQL Server
conn = psycopg2.connect(
   **config["postgres"]
)
postGrescursor = conn.cursor()

#..........................................................................................................................
#----------------- 2. Retrieve databases/schemas
# a. Get MySQL Dbs/
mySQLcursor.execute("show databases")
mysqlDbs = [x[0] for x in mySQLcursor if x[0] not in ('information_schema','deduplication','facility','performance_schema','mysql','mrs','provider','terminology')]

# b. Get PostgreSQL Dbs
postGrescursor.execute("SELECT nspname AS schema_name FROM pg_catalog.pg_namespace;")
postgresDbs = [x[0] for x in postGrescursor.fetchall() if x[0] not in ('pg_toast','pg_catalog','public','information_schema')]


#----------------3. Compare shemas
print(">>> Compare shemas")
if len(mysqlDbs) > len(postgresDbs):
    diff = len(mysqlDbs) - len(postgresDbs)
    for i in range(diff):
        postgresDbs.append("")
else:
    diff = len(postgresDbs) - len(mysqlDbs)
    for i in range(diff):
        mysqlDbs.append("")

# b. Create a dataframe of databases from facility and those from central repository
df_schema = pd.DataFrame({"Databases at Facility":mysqlDbs,"Databases at Central Repository":postgresDbs})


#-----------------b. Creating word document and create title
doc = docx.Document()
title = doc.add_heading('Quality Assessment Testing for Data Pipeline', 1)
title.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.CENTER
h1 = doc.add_heading('Prepared on :'+ str(date.today()), 2)
h1.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.CENTER
date_title = doc.add_paragraph()

#------------------c. Add database comparison to word
doc.add_heading("1. Databases Comparison",2)
doc_table = doc.add_table(df_schema.shape[0]+1, df_schema.shape[1])
doc_table.style = 'Light Grid Accent 5'
for j in range(df_schema.shape[-1]):
    doc_table.cell(0,j).text = df_schema.columns[j]
for i in range(df_schema.shape[0]):
    for j in range(df_schema.shape[-1]):
        doc_table.cell(i+1,j).text = str(df_schema.values[i,j])
doc.add_paragraph()
doc.add_heading("Comment",3)
if "" in mysqlDbs:
    mysqlDbs.remove("")

if "" in postgresDbs:
    postgresDbs.remove("")

if len(mysqlDbs) > len(postgresDbs):
    doc.add_paragraph("""There is a difference in number of databases at facility with that at central repository. 
                      """ )
    diff = set(mysqlDbs) - set(postgresDbs)
    if len(diff) > 1 :
        doc.add_paragraph("The following databases are not found at central repository:")
    else:
        doc.add_paragraph("The following database is not found at central repository:")

    for item in diff:
        doc.add_paragraph(item, style='List Bullet 2')
else:
    doc.add_paragraph("""All the databases at the facility have been successfully migrated to the central repository.
                      """ )


# ----------------- 4. Compare tables
print(">>> Compare tables")
# ------------------a. Compare tables only in common databases
common_databases = [i for i in mysqlDbs if i in postgresDbs]

flag = True 
database_dictionary_mysql = {}
database_dictionary_postgres = {}
number_of_tables_facility = [] 
number_of_tables_central = []
for db in common_databases:
    mySQLcursor.execute("USE " + db)
    mySQLcursor.execute("SHOW TABLES")
    tables = mySQLcursor.fetchall()
    mySqlTables = [x[0] for x in tables]
    number_of_tables_facility.append(len(mySqlTables))
    postGrescursor.execute("select * from pg_tables where schemaname= " +"\'" + db + "\'" + ";")
    postgresTables = [x[1] for x in postGrescursor.fetchall()]
    number_of_tables_central.append(len(postgresTables))
    database_dictionary_mysql[db] = mySqlTables
    database_dictionary_postgres[db] = postgresTables
df_tables = pd.DataFrame({"Database":common_databases,"Number of Tables at Facility":number_of_tables_facility,
                          "Number of Tables at Central repository":number_of_tables_central})

#------------------b. Add table comparison to word
doc.add_heading("2. Tables Comparison",2)
doc_table = doc.add_table(df_tables.shape[0]+1, df_tables.shape[1])
doc_table.style = 'Light Grid Accent 5'
for j in range(df_tables.shape[-1]):
    doc_table.cell(0,j).text = df_tables.columns[j]
for i in range(df_tables.shape[0]):
    for j in range(df_tables.shape[-1]):
        doc_table.cell(i+1,j).text = str(df_tables.values[i,j])

#--------------------c. Find tables that are not in central repo and vice versa
tables_not_in_central = set(mySqlTables) - set(postgresTables)
tables_not_at_facility = set(postgresTables) - set(mySqlTables)

# check if all tables at facility are in central repo and all in central repo are at facility
if not tables_not_in_central:
    if not tables_not_at_facility:
         doc.add_paragraph("All the tables at facility have been successfully migrated to the central repository.")

if tables_not_in_central:
    if tables_not_at_facility:
        doc.add_paragraph("The following tables are at facility but not in central repository.")
        for table_item in tables_not_in_central:
                doc.add_paragraph(table_item, style='List Bullet')
        doc.add_paragraph("The following tables are in central repository are not at facility.")
        for table_item in tables_not_in_central:
                doc.add_paragraph(table_item, style='List Bullet')
    else:
        doc.add_paragraph("The following tables are at facility but not in central repository.")
        for table_item in tables_not_in_central:
                doc.add_paragraph(table_item, style='List Bullet')

if tables_not_at_facility:
        doc.add_paragraph("""All the tables at the facility have been successfully migrated to the central repository. However, there are tables in the central repository that are not at the facility.
                          """)
        for table_item in tables_not_at_facility:
            doc.add_paragraph(table_item, style='List Bullet')
                                                                                                                                                                                        

# Compare columns
print(">>> Compare columns")
doc.add_heading("3. Columns Comparison",2)
for db in  sorted(database_dictionary_mysql.keys()):
     tables_mysql = database_dictionary_mysql[db]
     tables_postgres = database_dictionary_postgres[db]
     common_tables = set(tables_mysql).intersection(set(tables_postgres))
     for table in common_tables:
          mySQLcursor.execute("USE " + db)
          mySQLcursor.execute("SHOW columns FROM "+ table)
          mySQL_results = mySQLcursor.fetchall()
          # Get column names for both mysql and postgresql
          column_names_mysql = [x[0] for x in mySQL_results]
          #   data_types_mysql = [x[1] for x in mySQL_results]
          postGrescursor.execute("SELECT column_name, data_type, is_nullable FROM information_schema.columns WHERE table_name =  " +"\'" + table + "\'" + ";")
          postgresql_results = postGrescursor.fetchall()
          column_names_postgresql = [x[0] for x in postgresql_results if x[0] not in ('tenant_id','deleted')]
          # data_types_mysql = [x[1].decode('utf-8') for x in mySQL_results]
          # Check if column names are the same as those at facility 
        #   column_names_postgresql = column_names_postgresql.difference_update({})
          if set(column_names_mysql) - set(column_names_postgresql):
                if set(column_names_postgresql) - set(column_names_mysql):
                    doc.add_heading("Database:" + db + "  > Table: "+ table , 3)
                    doc.add_paragraph("The table "+ table + " has different column names at the facility compared to those at the central repository.",style="List Bullet 2")
                    doc.add_paragraph("The following columns are at facility but not in central repository")
                    for item in set(column_names_mysql) - set(column_names_postgresql):
                        doc.add_paragraph(item,style = "List Bullet 3")
                    doc.add_paragraph("The following columns are at central repository but not at facility")
                    for item in set(column_names_postgresql) - set(column_names_mysql):
                        doc.add_paragraph(item,style = "List Bullet 3")
          if set(column_names_mysql) - set(column_names_postgresql):
                doc.add_heading("Database:" + db + "  > Table: "+ table , 3)
                doc.add_paragraph("The following columns are at facility but not in central repository")
                for item in set(column_names_mysql) - set(column_names_postgresql):
                    doc.add_paragraph(item,style = "List Bullet 3")
          if set(column_names_postgresql) - set(column_names_mysql):
                doc.add_heading("Database:" + db + "  > Table: "+ table , 3)
                doc.add_paragraph("The following columns are at central repository but not at facility")
                for item in set(column_names_postgresql) - set(column_names_mysql):
                    doc.add_paragraph(item,style = "List Bullet 3")

print(">>> Table contents analysis")
doc.add_heading("4. Table contents Analysis",2)
database = []
table_name = []
number_of_records_in_table = []
number_of_records_in_central = []
number_of_missing_records = []
reason = []
percentage_variance = []
records_inc = []
differences = []
percentage_inconsistence = []
list_id = []
ids_with_inconsistencies = []
id_with_inconsistencies = {}
for db in database_dictionary_mysql.keys():
     tables_mysql = database_dictionary_mysql[db]
     tables_postgres = database_dictionary_postgres[db]
     common_tables = set(tables_mysql).intersection(set(tables_postgres))
     flag = True
     check_flag = True
     count_count = 1
     print("   >>> Working on", db)
     for table in common_tables:
        print(table)
        # get table contents from mysql
        mySQLcursor.execute("USE " + db)
        mySQLcursor.execute("SHOW columns FROM " + table)
        mySQL_results = mySQLcursor.fetchall()
        column_names_mysql = [x[0] for x in mySQL_results if x[0] not in ('option','procedure')]
        # get table contents from postgresql
        # Get the column names and data types for the table
        postGrescursor.execute("SELECT column_name, data_type, is_nullable FROM information_schema.columns WHERE table_name =  " +"\'" + table + "\'" + ";")
        postgresql_results = postGrescursor.fetchall()
        column_names_postgresql = [x[0] for x in postgresql_results if x[0] not in ('tenant_id','deleted')]
        common_columns = [i for i in column_names_mysql if i in column_names_postgresql]

        # mysql contents
        sql_query = ','.join(common_columns)
        query = "select "+ sql_query + " from "+ table
        mySQLcursor.execute(query)
        mySQL_results_contents = mySQLcursor.fetchall()
        
        date_columns_postresql = []
        # Iterate over the column information and identify the date columns
        for column in postgresql_results:
            if column[0] not in common_columns:
                 continue
            column_name = column[0]
            data_type = column[1]
            if data_type == 'date' or data_type == 'timestamp' or data_type == 'timestamp without time zone':
                date_columns_postresql.append(column_name)
        # identify non date columns
        non_date_columns = [x[0] for x in common_columns if x not in date_columns_postresql] 
        # construct query dynamically
        sql_q =  common_columns[0]
        # postGrescursor.execute("select " + sql_query + " from " + db + "." + table + 
        #                        " where tenant_id = " +"\'" + inputFacilityID + "\'" + ";"  )
        postGrescursor.execute("select " + sql_query + " from " + db + "." + table + 
                               " where tenant_id = " +"\'" + inputFacilityID + "\'"  + ";")
        
        postgres_results_contents = []
        try:
            postgres_results_contents = [x for x in postGrescursor.fetchall()]
        except Exception as e:
             postGrescursor.execute("select " + sql_q + " from " + db + "." + table + 
                               " where tenant_id = " +"\'" + inputFacilityID + "\'"  + ";")
             post_list = postGrescursor.fetchall()
             post_list = [x[0] for x in post_list]
             failed_list = []
             error_list = []
             counter = 0
             for id in post_list:
                  counter += 1
                  postGrescursor.execute("select "  +  sql_query + " from " + db + "." + table + 
                               " where tenant_id = " +"\'" + inputFacilityID + "\'"  +  " and " + sql_q + " = " +"\'" + id + "\'" +";")
                  try:
                        df_row = postGrescursor.fetchone()
                        postgres_results_contents.append(df_row)
                        continue
                  except Exception as e:
                        failed_list.append(id)
                        error_list.append(e)
                        doc.add_heading("Database:" + db + "  > Table: "+ table , 4)
                        doc.add_paragraph(sql_q + " >" + id + "  Reason of failure >" + str(e))
                        continue 
       
        diff  = set(mySQL_results_contents) - set(postgres_results_contents)
        differences.append(len(diff))
        reason_string = ""
        database.append(db)
        table_name.append(table)
        number_of_records_in_table.append(len(mySQL_results_contents))

        # Number of records that fail to be moved to srever
        key_mysql = [x[0] for x in mySQL_results_contents]
        key_postgresql = [x[0] for x in postgres_results_contents]
        diff_num_set = set(key_mysql) - set(key_postgresql)
        diff_num = len(diff_num_set)
        number_of_missing_records.append(diff_num)
        num = len(mySQL_results_contents) - diff_num
        number_of_records_in_central.append(num)     
        if len(mySQL_results_contents) == 0:
            perc_variance = 0
        else:
            perc_variance = (diff_num/ len(mySQL_results_contents)) * 100
            perc_variance = round(perc_variance,4)
        percentage_variance.append(perc_variance)

        # Number of records that have been moved to central repository but are inconsistent
        records_id_moved_to_central = set(key_mysql).intersection(set(key_postgresql))
        records_moved_to_central = [i for i in mySQL_results_contents if i[0] in records_id_moved_to_central]
        records_with_inconsistencies = set(records_moved_to_central) - set(postgres_results_contents)
        records_inc.append(len(records_with_inconsistencies))

        if num == 0:
            perc_incons = 0
        else:
            perc_incons = (len(records_with_inconsistencies) /num) * 100
            perc_incons = round(perc_incons,4)
        percentage_inconsistence.append(perc_incons)


        if len(records_with_inconsistencies) > 0:
            # determine reasons for differences
            reason_list = []
            id_list = []
            for element in records_with_inconsistencies:     
                list1 = [item for item in postgres_results_contents if item[0] == element[0]]
                res_str = []
                id_id_list = []
                for i in range(len(list1[0])):
                    if element[i] != list1[0][i]:                                                                                                       
                        res = column_names_mysql[i]
                        res_str.append(res)
                        id_id_list.append(element[0])
                for tt in res_str:
                    if tt not in reason_list:
                        reason_list.append(tt)
                for tt in id_id_list:
                    if tt not in id_list:
                        id_list.append(tt)
            res_string = ','.join(res_str)
            reason.append(res_string)
            list_id.append(','.join(id_list))
        else:
            reason.append("")
            list_id.append("")
        if len(records_with_inconsistencies)> 0:
            _ids = [x[0] for x in records_with_inconsistencies]
            id_with_inconsistencies[table] = _ids
            ids_with_inconsistencies.append(_ids)
        else:
            ids_with_inconsistencies.append("")

print(">>> Constructing dataframe")
df = pd.DataFrame({"Database Name":database,"Table Name":table_name,
                   "Number of records in facility database":number_of_records_in_table,
                   "Number of records in central database":number_of_records_in_central,
                   "Difference in records":number_of_missing_records,
                   "% Difference":percentage_variance,
                   "Records with inconsistencies": records_inc,
                   "% of inconsistence records":percentage_inconsistence,
                   "The following columns contain inconsistencies in data":reason,
                    "Id's with inconsistencies": ids_with_inconsistencies
                  })


standard_dev = df['Difference in records'].std() 
df = df.sort_values(by=['Difference in records'], ascending=False)
df.loc[""] = df.sum()
df.at[df.index[-1], 'Database Name'] = "Total"
df.at[df.index[-1], 'Table Name'] = ""
df.at[df.index[-1], '% Difference'] = (df.at[df.index[-1], 'Difference in records'] / df.at[df.index[-1], 'Number of records in facility database']) * 100


# print(">>> Adding results to excel rows")
# t = doc.add_table(df.shape[0]+1, df.shape[1])
# t.style = 'Light Grid Accent 5'
# for j in range(df.shape[-1]):
#     t.cell(0,j).text = df.columns[j]


# for i in range(df.shape[0]):
#     for j in range(df.shape[-1]):
#         t.cell(i+1,j).text = str(df.values[i,j])



print(">>> Saving file")
filename = "QAT " + inputFacilityID  +  " " + str(date.today()) + ".docx"
filename_csv = "QAT " + inputFacilityID  +  " " + str(date.today()) + ".csv"
df.to_csv(filename_csv)
doc.save(filename)

>>> Compare shemas
>>> Compare tables
>>> Compare columns
>>> Table contents analysis
   >>> Working on client
email
identification
relationship
phone
person
   >>> Working on consultation
dog_bite_vaccination_status
screen_corona_virus
essential_baby_care_medicine
bin_test_kit_order
stock_bin_batch_issue
art_who_stage
fetal_heart_rate
person_investigation
muac
sti
tb_clinical_assessment
malaria_suspected
cbs
tb_directly_observed_treatment
tb_treatment_current_status
art_linkage_from
vagina_monitoring
cervical_cancer_screening
drug_adverse_event
hts_investigation
patient_procedure
medicine_batch_issue
tb_treatment_planning
partogram_reading_contraction
cervical_cancer_screening_history
prep_risk_assessment
art_ipt
hts_screening_question
prep_initiation
infant_eid
prep_visit
patient_family_planning
client_port_of_entry
umbilical_cord
still_birth_defect
art_transfer_out
appointment
dna_pcr_test
head_circumference
sundry_batch
sexual_history
pregnancy_live_birth
sundry_user_issue
bin_test

C:\Users\Simba\AppData\Local\Temp\ipykernel_26828\1972341329.py:371: FutureWarning: The default value of numeric_only in DataFrame.sum is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  df.loc[""] = df.sum()


PermissionError: [Errno 13] Permission denied: 'QAT ZW000B03 2023-11-01.csv'